# BirdCLEF+ 2026 EfficientNet-B0

Train a 5-second mel-spectrogram EfficientNet-B0 baseline and generate `submission.csv` in the same Kaggle run. This is the primary competition-safe workflow.


# 1. Train EfficientNet-B0

The model uses primary labels, stratified folds, lightweight augmentation, and single-process audio loading for stable Kaggle execution.


## 1.1 Environment


In [ ]:
from pathlib import Path
import json
import os
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def read_optional_csv(path: Path) -> pd.DataFrame | None:
    return pd.read_csv(path) if path.exists() else None


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Output directory: {CFG.artifact_dir}")

try:
    import timm
except ImportError:
    import sys
    !{sys.executable} -m pip install -q timm
    import timm

import librosa
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.benchmark = True
torch.set_num_threads(min(2, os.cpu_count() or 1))
try:
    torch.multiprocessing.set_sharing_strategy("file_system")
except RuntimeError:
    pass


class CFG(CFG):
    artifact_dir = Path("/kaggle/working/artifacts/effnet_b0")
    sample_rate = 32000
    duration = 5.0
    n_fft = 2048
    hop_length = 512
    n_mels = 128
    fmin = 20
    fmax = 16000
    n_splits = 5
    fold = 0
    backbone = "efficientnet_b0"
    pretrained = True
    epochs = 5
    batch_size = 32
    # Kaggle notebooks can crash with multiprocessing DataLoader workers when
    # audio decoding happens inside __getitem__. Keep this at 0 for stability.
    num_workers = 0
    force_single_process_loader = True
    lr = 3e-4
    weight_decay = 1e-2
    label_smoothing = 0.05
    max_train_samples = None
    max_valid_samples = None


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
TRAIN_ARTIFACT_DIR = CFG.artifact_dir
SUBMISSION_ARTIFACT_DIR = Path("/kaggle/working/artifacts/submission")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 1.2 Metadata And Folds


Build the label map and validation fold from `train.csv`. The fold file is saved with the artifacts so model results can be traced back to the exact split.


In [ ]:
train = pd.read_csv(CFG.data_root / "train.csv")
train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
labels = sorted(train["primary_label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["target"] = train["primary_label"].map(label_to_idx)

train["fold"] = -1
group_col = "filename"
try:
    splitter = StratifiedGroupKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
    splits = splitter.split(train, train["target"], groups=train[group_col])
except ValueError:
    splitter = StratifiedKFold(n_splits=CFG.n_splits, shuffle=True, random_state=CFG.seed)
    splits = splitter.split(train, train["target"])

for fold, (_, valid_idx) in enumerate(splits):
    train.loc[valid_idx, "fold"] = fold

train.to_csv(CFG.artifact_dir / "train_folds.csv", index=False)
(CFG.artifact_dir / "labels.json").write_text(json.dumps(idx_to_label, indent=2), encoding="utf-8")

train_df = train[train["fold"] != CFG.fold].reset_index(drop=True)
valid_df = train[train["fold"] == CFG.fold].reset_index(drop=True)
if CFG.max_train_samples:
    train_df = train_df.sample(CFG.max_train_samples, random_state=CFG.seed).reset_index(drop=True)
if CFG.max_valid_samples:
    valid_df = valid_df.sample(CFG.max_valid_samples, random_state=CFG.seed).reset_index(drop=True)

print(f"Train rows: {len(train_df):,}")
print(f"Valid rows: {len(valid_df):,}")
print(f"Classes: {len(labels):,}")


## 1.3 Dataset


Each recording becomes a normalized mel-spectrogram. Training uses random gain augmentation; validation uses deterministic 5-second crops for a stable reference metric.


In [ ]:
def load_audio(path: Path, duration: float, train_mode: bool) -> np.ndarray:
    target_len = int(CFG.sample_rate * duration)
    offset = 0.0
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, offset=offset, duration=duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def audio_to_mel(y: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=CFG.sample_rate,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        n_mels=CFG.n_mels,
        fmin=CFG.fmin,
        fmax=CFG.fmax,
        power=2.0,
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)


class BirdDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train_mode: bool):
        self.df = df.reset_index(drop=True)
        self.train_mode = train_mode

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        y = load_audio(row["filepath"], CFG.duration, self.train_mode)
        if self.train_mode and random.random() < 0.5:
            y = y * random.uniform(0.75, 1.25)
        x = torch.from_numpy(audio_to_mel(y)).unsqueeze(0)
        target = torch.tensor(row["target"], dtype=torch.long)
        return x, target


def effective_num_workers() -> int:
    if CFG.force_single_process_loader or Path("/kaggle/working").exists():
        return 0
    return int(CFG.num_workers)


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    workers = effective_num_workers()
    loader_kwargs = {
        "batch_size": batch_size,
        "shuffle": shuffle,
        "num_workers": workers,
        "pin_memory": device.type == "cuda",
        "drop_last": False,
    }
    if workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 2
    return DataLoader(dataset, **loader_kwargs)


print(f"Effective DataLoader workers: {effective_num_workers()}")
train_loader = make_loader(BirdDataset(train_df, train_mode=True), CFG.batch_size, True)
valid_loader = make_loader(BirdDataset(valid_df, train_mode=False), CFG.batch_size * 2, False)


## 1.4 Model


EfficientNet-B0 is small enough for fast Kaggle reruns while still giving a useful acoustic CNN baseline for comparison against Perch features.


In [ ]:
class BirdClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.model = timm.create_model(
            CFG.backbone,
            pretrained=CFG.pretrained,
            in_chans=1,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.model(x)


model = BirdClassifier(num_classes=len(labels)).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")


In [ ]:
def train_one_epoch() -> float:
    model.train()
    total_loss = 0.0
    for x, y in tqdm(train_loader, desc="train", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.size(0)
    return total_loss / max(len(train_loader.dataset), 1)


@torch.no_grad()
def validate() -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    for x, y in tqdm(valid_loader, desc="valid", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        loss = criterion(logits, y)
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        seen += x.size(0)
    return total_loss / max(seen, 1), correct / max(seen, 1)


## 1.5 Training


Save the checkpoint with the best validation accuracy and record the epoch-level learning curve in `history.csv`.


In [ ]:
history = []
best_acc = 0.0

for epoch in range(1, CFG.epochs + 1):
    train_loss = train_one_epoch()
    valid_loss, valid_acc = validate()
    scheduler.step()
    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
        "lr": scheduler.get_last_lr()[0],
    }
    history.append(row)
    print(row)

    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(
            {
                "model": model.state_dict(),
                "label_to_idx": label_to_idx,
                "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                "valid_acc": best_acc,
            },
            CFG.artifact_dir / "best_effnet_b0.pt",
        )

history_df = pd.DataFrame(history)
history_df.to_csv(CFG.artifact_dir / "history.csv", index=False)
print(f"Best valid accuracy: {best_acc:.4f}")
print(f"Saved outputs to {CFG.artifact_dir}")


## 1.6 Save Training Outputs


Save the checkpoint, label map, fold file, and training history as a compact Kaggle output bundle.


In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = Path("/kaggle/working/birdclef_effnet_b0_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(zip_path))


# 2. Generate Submission

Use the newly trained checkpoint, or an attached EfficientNet checkpoint, to score the test soundscape windows expected by `sample_submission.csv`.


## 2.1 Inference Environment


In [ ]:
import zipfile
from IPython.display import FileLink

SUBMISSION_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CFG.inference_batch_size = 32
CFG.checkpoint_path = None
CFG.labels_path = None

print(f"Device: {device}")
print(f"Training output directory: {TRAIN_ARTIFACT_DIR}")
print(f"Submission output directory: {SUBMISSION_ARTIFACT_DIR}")


## 2.2 Competition Files And Checkpoint


Locate `sample_submission.csv`, `best_effnet_b0.pt`, and `labels.json`. The search prefers the checkpoint produced by this run, then falls back to attached Kaggle inputs.


In [ ]:
def find_file(filename: str, roots: list[Path]) -> Path:
    for root in roots:
        if not root.exists():
            continue
        direct = root / filename
        if direct.exists():
            return direct
        matches = list(root.glob(f"**/{filename}"))
        if matches:
            return matches[0]
    searched = ", ".join(str(root) for root in roots)
    raise FileNotFoundError(f"Could not find {filename}. Searched: {searched}")


artifact_roots = [
    TRAIN_ARTIFACT_DIR,
    Path("/kaggle/input"),
]
sample_path = CFG.data_root / "sample_submission.csv"
checkpoint_path = Path(CFG.checkpoint_path) if CFG.checkpoint_path else find_file("best_effnet_b0.pt", artifact_roots)
labels_path = Path(CFG.labels_path) if CFG.labels_path else find_file("labels.json", artifact_roots)

sample_submission = pd.read_csv(sample_path)
idx_to_label = {int(k): v for k, v in json.loads(labels_path.read_text()).items()}
labels = [idx_to_label[i] for i in sorted(idx_to_label)]
label_to_idx = {label: i for i, label in enumerate(labels)}
target_columns = [c for c in sample_submission.columns if c != "row_id"]

print(f"Sample submission: {sample_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Labels: {labels_path}")
print(f"Rows: {len(sample_submission):,}")
print(f"Submission target columns: {len(target_columns):,}")
print(f"Model classes: {len(labels):,}")
display(sample_submission.head())


## 2.3 Model And Audio Index


In [ ]:
class BirdClassifier(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.model = timm.create_model(
            CFG.backbone,
            pretrained=False,
            in_chans=1,
            num_classes=num_classes,
        )

    def forward(self, x):
        return self.model(x)


def torch_load(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=device)


checkpoint = torch_load(checkpoint_path)
model = BirdClassifier(num_classes=len(labels)).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()


def row_to_stem_and_end_time(row_id: str) -> tuple[str, float]:
    stem, sep, end = str(row_id).rpartition("_")
    if sep and end.replace(".", "", 1).isdigit():
        return stem, float(end)
    return str(row_id), CFG.duration


def build_audio_index() -> dict[str, Path]:
    candidates = [
        CFG.data_root / "test_soundscapes",
        CFG.data_root / "test_audio",
        CFG.data_root / "train_soundscapes",
    ]
    audio_index = {}
    for folder in candidates:
        if folder.exists():
            for ext in ("*.ogg", "*.wav", "*.flac", "*.mp3"):
                for path in folder.glob(ext):
                    audio_index[path.stem] = path
    return audio_index


audio_index = build_audio_index()
print(f"Indexed audio files: {len(audio_index):,}")


## 2.4 Inference


Parse each `row_id` as a 5-second window ending at its timestamp, convert the segment to a mel-spectrogram, and write class probabilities into the submission columns. Tiny public dry-runs without mounted audio keep the sample submission values; hidden/test runs use model probabilities when audio is available.


In [ ]:
def load_audio_segment(path: Path, end_time: float) -> np.ndarray:
    offset = max(0.0, float(end_time) - CFG.duration)
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, offset=offset, duration=CFG.duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def audio_to_mel(y: np.ndarray) -> np.ndarray:
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=CFG.sample_rate,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        n_mels=CFG.n_mels,
        fmin=CFG.fmin,
        fmax=CFG.fmax,
        power=2.0,
    )
    mel = librosa.power_to_db(mel, ref=np.max)
    mel = (mel - mel.mean()) / (mel.std() + 1e-6)
    return mel.astype(np.float32)


def predict_batch(batch: list[np.ndarray]) -> np.ndarray:
    x = torch.from_numpy(np.stack(batch)).unsqueeze(1).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
    return probs


submission = sample_submission.copy()
for col in target_columns:
    submission[col] = 0.0

missing_audio = []
public_dry_run = len(audio_index) == 0 and len(submission) <= 3

if public_dry_run:
    print("No test audio found for the tiny public dry-run; preserving sample submission values.")
    submission = sample_submission.copy()
else:
    batch = []
    batch_rows = []

    for row_idx, row_id in tqdm(list(enumerate(submission["row_id"])), desc="inference"):
        stem, end_time = row_to_stem_and_end_time(row_id)
        audio_path = audio_index.get(stem)
        if audio_path is None:
            missing_audio.append(row_id)
            continue
        batch.append(audio_to_mel(load_audio_segment(audio_path, end_time)))
        batch_rows.append(row_idx)
        if len(batch) == CFG.inference_batch_size:
            probs = predict_batch(batch)
            for local_idx, submit_idx in enumerate(batch_rows):
                for label, class_idx in label_to_idx.items():
                    if label in target_columns:
                        submission.loc[submit_idx, label] = probs[local_idx, class_idx]
            batch = []
            batch_rows = []

    if batch:
        probs = predict_batch(batch)
        for local_idx, submit_idx in enumerate(batch_rows):
            for label, class_idx in label_to_idx.items():
                if label in target_columns:
                    submission.loc[submit_idx, label] = probs[local_idx, class_idx]

missing_audio_path = SUBMISSION_ARTIFACT_DIR / "missing_test_audio.json"
missing_audio_path.write_text(json.dumps(missing_audio, indent=2), encoding="utf-8")
print(f"Missing audio rows: {len(missing_audio):,}")
display(submission.head())


## 2.5 Submission Checks

- Confirm `submission.csv` has the same rows and target columns as `sample_submission.csv`.
- Review `missing_test_audio.json`; hidden-test runs should have no missing audio rows.
- Use the validation history as a sanity check, not as a complete proxy for leaderboard score.


## 2.6 Save Submission


In [ ]:
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
submission.to_csv(SUBMISSION_ARTIFACT_DIR / "submission.csv", index=False)

zip_path = Path("/kaggle/working/birdclef_effnet_b0_submission_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_path, arcname="submission.csv")
    for path in sorted(SUBMISSION_ARTIFACT_DIR.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(SUBMISSION_ARTIFACT_DIR.parent))

print(f"Wrote submission: {submission_path}")
print(f"Wrote artifact zip: {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(submission_path))
display(FileLink(zip_path))
